[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_cleaning.ipynb)

# DataSage Stage 1: Cleaning GRPO Training

Pre-builds dataset with real environment observations (no `rollout_func`).
Reward functions replay the env with stored seeds for context-matched evaluation.

**Fully self-contained** — runs in Colab with no repo clone or project imports.

**Requirements:** GPU (A100/H100), `HF_TOKEN`, `WANDB_API_KEY` env vars.

In [1]:
# Setup: install dependencies
!pip install -q unsloth trl datasets wandb requests vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.0/322.0 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0

In [2]:
# Load API keys from Colab Secrets or set manually
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

Loaded keys from Colab Secrets
Keys loaded.


In [3]:
# Config + parser + reward helpers (all inlined, no project imports)
import json
import re
import requests

# ---- Config constants ----
WANDB_PROJECT = "datasage"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
ENV_URL = "https://ricalanis-datasage-cleaning.hf.space"
HF_REPO = "ricalanis/datasage-qwen-cleaning"

LEARNING_RATE = 5e-6
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
NUM_GENERATIONS = 8
MAX_COMPLETION_LENGTH = 256
MAX_PROMPT_LENGTH = 1024


# ---- Completion text extraction ----
def _get_text(completion) -> str:
    """Extract text from a completion (str or chat message list)."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        # Chat format: [{"role": "assistant", "content": "..."}]
        return completion[-1]["content"] if completion else ""
    return str(completion)


# ---- Parser ----
def _extract_column(text: str) -> str:
    """Try to extract a column name from text."""
    quoted = re.findall(r'["\']([\w]+)["\']', text)
    if quoted:
        return quoted[0]
    camel = re.findall(r'\b([A-Z][a-z]+(?:[A-Z][a-z]+)+)\b', text)
    if camel:
        return camel[0]
    return ""


def parse_cleaning_action(text: str) -> dict:
    """Parse model output into a cleaning action dict."""
    json_match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "operation" in data and "column" in data:
                return data
        except json.JSONDecodeError:
            pass

    text_lower = text.lower()
    if "fill" in text_lower or "null" in text_lower:
        col = _extract_column(text)
        return {"operation": "fill_null", "column": col, "value": "median", "params": {}}
    elif "type" in text_lower or "cast" in text_lower or "convert" in text_lower:
        col = _extract_column(text)
        return {"operation": "fix_type", "column": col, "value": "numeric", "params": {}}
    elif "duplicate" in text_lower or "dedup" in text_lower:
        return {"operation": "remove_duplicate", "column": "", "params": {}}
    elif "standard" in text_lower or "normalize" in text_lower:
        col = _extract_column(text)
        return {"operation": "standardize", "column": col, "params": {}}
    elif "trim" in text_lower or "whitespace" in text_lower:
        col = _extract_column(text)
        return {"operation": "trim", "column": col, "params": {}}
    elif "typo" in text_lower or "correct" in text_lower:
        col = _extract_column(text)
        return {"operation": "correct_typo", "column": col, "params": {}}

    return {"operation": "fill_null", "column": "", "value": "median", "params": {}}


# ---- Reward helpers ----
_ACTION_JSON_RE = re.compile(r'\{[^{}]*"operation"[^{}]*\}')
CLEANING_VALID_OPS = {"fill_null", "fix_type", "remove_duplicate",
                      "standardize", "trim", "correct_typo"}


def cleaning_json_format_reward(completions, **kwargs) -> list[float]:
    """Reward well-formed JSON cleaning actions."""
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        match = _ACTION_JSON_RE.search(text)
        if match:
            try:
                data = json.loads(match.group())
                if data.get("operation") in CLEANING_VALID_OPS and "column" in data:
                    rewards.append(1.0)
                elif data.get("operation") in CLEANING_VALID_OPS:
                    rewards.append(0.6)
                else:
                    rewards.append(0.3)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


print(f"Environment URL: {ENV_URL}")
print(f"Base model: {BASE_MODEL}")

Environment URL: https://ricalanis-datasage-cleaning.hf.space
Base model: unsloth/Qwen2.5-3B-Instruct


In [4]:
# Model loading via Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.6,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {BASE_MODEL}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 03-08 10:06:57 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit with actual GPU utilization = 59.17%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.49 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 80.
Unsloth: vLLM's KV Cache can use up to 

/usr/local/lib/python3.12/dist-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 03-08 10:07:22 [model.py:531] Resolved architecture: Qwen2ForCausalLM
INFO 03-08 10:07:22 [model.py:1554] Using max model len 2048
INFO 03-08 10:07:23 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=6144.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'bfloat16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.2.mlp', 'model.layers.3.mlp', 'model.layers.30.mlp'], 'llm_int8_threshold': 6.0}
INFO 03-08 10:07:23 [vllm.py:747] Asynchronous scheduling is enabled.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded: unsloth/Qwen2.5-3B-Instruct


In [5]:
# System prompt and task descriptions
SYSTEM_PROMPT = """\
You are a data quality agent. You clean enterprise datasets across multiple \
domains (HR, Sales, Project Management, IT Operations).

Analyze the data quality report and data preview below, then respond with a \
JSON cleaning action:
{"operation": "<op>", "column": "<col>", "value": "<val>", "params": {}}

Available operations:
- fill_null: Fill null values (value can be "median", "mode", or a specific value)
- fix_type: Fix type mismatches in a column (cast to proper type)
- remove_duplicate: Remove duplicate rows
- standardize: Standardize column values (strip whitespace, normalize case)
- trim: Trim whitespace from column values
- correct_typo: Correct typos in categorical values

Identify the most impactful issue and act."""

TASK_DESCRIPTIONS = [
    "Clean the data to maximize the data quality score.",
    "Fix the most impactful data quality issue in this dataset.",
    "Analyze the DQ report and apply the best cleaning operation.",
    "This data has quality issues. Identify and fix the worst one.",
    "Improve the data quality score as much as possible with one operation.",
    "Look at the null counts, type issues, and duplicates. Fix the biggest problem.",
    "The data quality score is low. Apply the cleaning operation with the highest impact.",
    "Examine the data preview and quality report. What single operation improves quality most?",
]

In [6]:
# Dataset: pre-built with real environment observations
import random
from datasets import Dataset

random.seed(42)
SEEDS = [42 + i for i in range(64)]
prompts = []

print("Building dataset with real environment observations...")
for i, seed in enumerate(SEEDS):
    task_desc = random.choice(TASK_DESCRIPTIONS)
    resp = requests.post(f"{ENV_URL}/web/reset", json={"seed": seed})
    resp.raise_for_status()
    obs = resp.json()["observation"]

    prompts.append([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Domain: {obs['domain']}\n\n"
            f"Data Quality Report:\n{obs['dq_report']}\n"
            f"DQ Score: {obs['dq_score']:.4f}\n\n"
            f"Columns:\n{obs['columns_info']}\n\n"
            f"Data Preview:\n{obs['data_preview']}\n\n"
            f"Task: {task_desc}"
        )},
    ])
    if (i + 1) % 16 == 0:
        print(f"  Built {i + 1}/64 examples")

dataset = Dataset.from_dict({
    "prompt": prompts,
    "seed": SEEDS,
})
print(f"Dataset size: {len(dataset)}")

Building dataset with real environment observations...
  Built 16/64 examples
  Built 32/64 examples
  Built 48/64 examples
  Built 64/64 examples
Dataset size: 64


In [7]:
# Environment reward function (calls env directly with stored seeds)
def cleaning_env_reward(completions, **kwargs) -> list[float]:
    """Primary reward: replay env with stored seed, step with parsed action."""
    seeds = kwargs.get("seed", [0] * len(completions))
    rewards = []
    for completion, seed in zip(completions, seeds):
        try:
            text = _get_text(completion)
            action_dict = parse_cleaning_action(text)
            requests.post(f"{ENV_URL}/web/reset", json={"seed": int(seed)})
            resp = requests.post(f"{ENV_URL}/web/step", json={"action": action_dict})
            resp.raise_for_status()
            rewards.append(float(resp.json().get("reward", 0.0)))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

In [8]:
# GRPO training config
from trl import GRPOConfig

training_args = GRPOConfig(
    output_dir="./outputs/cleaning-grpo",
    use_vllm=True,
    learning_rate=LEARNING_RATE,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_grad_norm=0.1,
    logging_steps=1, save_steps=50,
    report_to="wandb", run_name="datasage-cleaning-grpo-v2",
)
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


'datasage'

In [9]:
# Create trainer and train
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args,
    train_dataset=dataset,
    reward_funcs=[cleaning_env_reward, cleaning_json_format_reward],
)
print("Starting Stage 1 (Cleaning) GRPO training...")
trainer.train()

Starting Stage 1 (Cleaning) GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 64 | Num Epochs = 3 | Total steps = 192
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ricalanis to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


WARNING 03-08 10:10:56 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / cleaning_env_reward / mean,rewards / cleaning_env_reward / std,rewards / cleaning_json_format_reward / mean,rewards / cleaning_json_format_reward / std
1,0.086500,1.212275,0.462596,32.125000,25.000000,39.000000,0.000000,32.125000,25.000000,39.000000,0,0,0,0,0,0.000000,0.962275,0.003121,0.250000,0.462910
2,0.268700,0.962650,0.003247,68.000000,29.000000,138.000000,0.000000,68.000000,29.000000,138.000000,No Log,No Log,No Log,No Log,No Log,-0.000000,0.962650,0.003247,0.000000,0.000000
3,-0.219800,1.087725,0.354688,35.750000,30.000000,58.000000,0.000000,35.750000,30.000000,58.000000,No Log,No Log,No Log,No Log,No Log,0.002315,0.962725,0.003313,0.125000,0.353553
4,-0.209600,0.961375,0.002425,43.375000,31.000000,73.000000,0.000000,43.375000,31.000000,73.000000,No Log,No Log,No Log,No Log,No Log,0.002097,0.961375,0.002425,0.000000,0.000000
5,0.027700,1.088250,0.354466,61.125000,31.000000,121.000000,0.000000,61.125000,31.000000,121.000000,No Log,No Log,No Log,No Log,No Log,0.002242,0.963250,0.001863,0.125000,0.353553
6,0.190100,0.964875,0.002150,44.500000,29.000000,90.000000,0.000000,44.500000,29.000000,90.000000,No Log,No Log,No Log,No Log,No Log,0.000441,0.964875,0.002150,0.000000,0.000000
7,0.080100,1.086863,0.353574,36.375000,28.000000,66.000000,0.000000,36.375000,28.000000,66.000000,No Log,No Log,No Log,No Log,No Log,0.001702,0.961863,0.002006,0.125000,0.353553
8,0.021200,0.961863,0.002006,48.875000,29.000000,150.000000,0.000000,48.875000,29.000000,150.000000,No Log,No Log,No Log,No Log,No Log,0.002051,0.961863,0.002006,0.000000,0.000000
9,-0.037600,0.959912,0.002864,33.500000,29.000000,39.000000,0.000000,33.500000,29.000000,39.000000,No Log,No Log,No Log,No Log,No Log,0.000574,0.959912,0.002864,0.000000,0.000000
10,0.012100,0.961700,0.004354,33.250000,28.000000,39.000000,0.000000,33.250000,28.000000,39.000000,No Log,No Log,No Log,No Log,No Log,0.000675,0.961700,0.004354,0.000000,0.000000


profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▄▃▁▁█▁▂▂▃▂▂▂▃▂▃▃▅▃▂▁▂▂▂▁▂▂▃▂▂▂▂▃▂▂▂▃▁▂▁▂
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁█▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
profiling/Time taken: UnslothGRPOTrainer.cleaning_env_reward,▃▇▅▆▂▃▅▂▄▅▃▂▅▃█▂▂▃▄▆▅▂▅▄▇▆▄▂▃▃▁▆▃▄▂▅▂▃▁▃
profiling/Time taken: UnslothGRPOTrainer.cleaning_json_format_reward,▅▃▄▁▂▂▂▃▂▃▃█▃▅▅▃▄▃▂▃▃▄▃▄▃▂▄▃▃▃▃▃▃▂▃▃▅▅▄▃
profiling/Time taken: UnslothGRPOTrainer.vLLM.generate,▂▇▁▁▁▁▂▂▂▁▂▁▁▁▁▁█▁▄▁▁▁▁▁▁▁▁▁▃▁▁▁▁▂▅▁▁▂▁▁
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_min,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/region_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+22,...


TrainOutput(global_step=192, training_loss=0.0687127578337936, metrics={'train_runtime': 1494.1224, 'train_samples_per_second': 0.129, 'train_steps_per_second': 0.129, 'total_flos': 0.0, 'train_loss': 0.0687127578337936})

In [10]:
# Save and push to Hub
output_dir = "./outputs/cleaning-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

print(f"Pushing to Hub: {HF_REPO}")
trainer.push_to_hub(HF_REPO)
print(f"Model pushed to https://huggingface.co/{HF_REPO}")

Model saved to ./outputs/cleaning-grpo-final
Pushing to Hub: ricalanis/datasage-qwen-cleaning


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...aning-grpo/tokenizer.json:  30%|###       | 3.46MB / 11.4MB            

  ...adapter_model.safetensors:   2%|2         | 2.79MB /  120MB            

  ...ng-grpo/training_args.bin:   4%|3         |   294B / 7.38kB            

Model pushed to https://huggingface.co/ricalanis/datasage-qwen-cleaning
